In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [ ]:
DATA_PATH = "data/combined_traces_dataset.csv"

In [ ]:
df = pd.read_csv(DATA_PATH)
print(df.shape)

In [ ]:
syscall_list_cols = [c for c in df.columns if c.startswith("syscall_") and c.endswith("_List")]
print(syscall_list_cols)

In [ ]:
def merge_syscalls(row):
    tokens = []
    for col in syscall_list_cols:
        val = row[col]
        if pd.isna(val):
            continue
        parts = [p.strip() for p in str(val).split(",") if p.strip()]
        tokens.extend(parts)
    return " ".join(tokens)

df["merged_syscalls"] = df.apply(merge_syscalls, axis=1)

In [ ]:
vectorizer = CountVectorizer(
    ngram_range=(2, 2),
    token_pattern=r"\b\w+\b",
    min_df=5
)

X_patterns = vectorizer.fit_transform(df["merged_syscalls"])
y = df["Level"]

print(X_patterns.shape)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_patterns,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [ ]:
rf = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)
pred = rf.predict(X_test)

In [ ]:
print("Accuracy:", accuracy_score(y_test, pred))
print("Precision:", precision_score(y_test, pred))
print("Recall:", recall_score(y_test, pred))
print("F1:", f1_score(y_test, pred))

In [ ]:
scores = cross_val_score(rf, X_patterns, y, cv=5)
print("Cross-validation accuracy:", scores.mean())